In [1]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False)

angle_triplets = [
    (11, 13, 15), (13, 11, 23), (11, 23, 25), (23, 25, 27), (25, 27, 31),
    (15, 13, 11), (27, 25, 23), (23, 11, 13), (13, 15, 17), (11, 13, 15),
    (12, 14, 16), (14, 12, 24), (12, 24, 26), (24, 26, 28), (26, 28, 32),
    (16, 14, 12), (28, 26, 24), (24, 12, 14), (14, 16, 18), (12, 14, 16),
    (11, 12, 0), (23, 24, 0), (0, 11, 23), (0, 12, 24),
    (15, 17, 19), (16, 18, 20),
    (0, 23, 24), (0, 11, 12), (0, 13, 14),
    (27, 31, 29), (28, 32, 30),
    (23, 24, 26)
]

def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

def get_32_angles(landmarks):
    angles = []
    for a, b, c in angle_triplets:
        try:
            a_coord = [landmarks[a].x, landmarks[a].y]
            b_coord = [landmarks[b].x, landmarks[b].y]
            c_coord = [landmarks[c].x, landmarks[c].y]
            angle = calculate_angle(a_coord, b_coord, c_coord)
        except:
            angle = 0.0
        angles.append(angle)
    return angles

# Process all labeled folders
data = []
root_folder = "exercise_videos"

for label in os.listdir(root_folder):
    path = os.path.join(root_folder, label)
    if not os.path.isdir(path):
        continue

    print(f"Processing: {label}")
    for file in os.listdir(path):
        cap = cv2.VideoCapture(os.path.join(path, file))
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(image)

            if results.pose_landmarks:
                angles = get_32_angles(results.pose_landmarks.landmark)
                data.append(angles + [label.lower()])
        cap.release()

# Save dataset
columns = [f'angle_{i+1}' for i in range(32)] + ['label']
df = pd.DataFrame(data, columns=columns)
df.to_csv("exercise_32angles_dataset.csv", index=False)
print("✅ Saved to exercise_32angles_dataset.csv")


Processing: bicepcurl
Processing: forward lunges
Processing: lateralraises
Processing: legraises
Processing: planks
Processing: pushups
Processing: shoulderpress
Processing: squats
✅ Saved to exercise_32angles_dataset.csv


In [2]:
df['label'].unique()


array(['bicepcurl', 'forward lunges', 'lateralraises', 'legraises',
       'planks', 'pushups', 'shoulderpress', 'squats'], dtype=object)

In [3]:
df['label'].value_counts()

label
forward lunges    2440
pushups           2271
shoulderpress     2125
squats            2121
planks            2119
lateralraises     2089
legraises         2058
bicepcurl         1856
Name: count, dtype: int64